# 3D Spatial Mapping for Crazyflie 2.0 Drone
## Real-time Occupancy Grid from Monocular B&W Camera

This notebook implements a complete spatial mapping pipeline:
- **Monocular depth estimation** (Depth Anything V2)
- **Visual odometry** (pose tracking)
- **3D occupancy grid** (voxel-based)
- **Real-time visualization**

Target: 30 FPS on cloud infrastructure for indoor navigation

## 1. Install Dependencies

In [1]:
!pip install torch torchvision opencv-python numpy scipy open3d transformers timm matplotlib scikit-image

  Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached timm-1.0.24-py3-none-any.whl.metadata (38 kB)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached dash-4.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached werkzeug-3.1.5-py3-none-any.whl.metadata (4.0 kB)
  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached configargparse-1.7.1-py3-none-any.whl.metadata (24 kB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyquaternion-0.9.9-py3-none-any.whl.metadata (1.4 kB)
  Using cached

## 2. Import Libraries

In [1]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import open3d as o3d
import time
from collections import deque
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R

print(f"PyTorch version: {torch.__version__}")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

/Users/tommy/Projects/drone/.drone_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0
Using device: mps


## 3. Load Depth Estimation Model (Depth Anything V2)

Using Depth Anything V2 Small for optimal speed/accuracy trade-off at 30 FPS

In [2]:
print("Loading Depth Anything V2 model...")
# Use the small model for 30 FPS performance
model_name = "depth-anything/Depth-Anything-V2-Small-hf"
image_processor = AutoImageProcessor.from_pretrained(model_name)
depth_model = AutoModelForDepthEstimation.from_pretrained(model_name)
depth_model = depth_model.to(device)
depth_model.eval()
print("✓ Depth model loaded successfully")

Loading Depth Anything V2 model...


The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 287/287 [00:00<00:00, 2066.28it/s, Materializing param=neck.reassemble_stage.layers.3.resize.weight]                  


✓ Depth model loaded successfully


## 4. Visual Odometry Class

Tracks camera pose using optical flow and feature matching

In [3]:
class VisualOdometry:
    """Simple visual odometry using feature tracking"""
    
    def __init__(self):
        # ORB feature detector
        self.detector = cv2.ORB_create(nfeatures=2000)
        self.matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
        
        # Camera intrinsics (approximate for webcam - will need calibration for real drone)
        self.focal_length = 500  # pixels
        self.cx = 320  # principal point x
        self.cy = 240  # principal point y
        
        # Pose tracking
        self.position = np.array([0.0, 0.0, 0.0])  # x, y, z in meters
        self.rotation = np.eye(3)  # rotation matrix
        
        # Previous frame data
        self.prev_frame = None
        self.prev_kp = None
        self.prev_desc = None
        
        # Movement scale (will be refined with depth)
        self.scale = 0.01  # meters per pixel movement
        
    def update(self, frame_gray, depth_map=None):
        """Update pose based on new frame"""
        
        # Detect features
        kp, desc = self.detector.detectAndCompute(frame_gray, None)
        
        if self.prev_frame is None:
            # First frame - initialize
            self.prev_frame = frame_gray.copy()
            self.prev_kp = kp
            self.prev_desc = desc
            return self.position.copy(), self.rotation.copy()
        
        # Match features between frames
        if desc is not None and self.prev_desc is not None and len(kp) > 10:
            matches = self.matcher.knnMatch(self.prev_desc, desc, k=2)
            
            # Apply ratio test
            good_matches = []
            for m_n in matches:
                if len(m_n) == 2:
                    m, n = m_n
                    if m.distance < 0.75 * n.distance:
                        good_matches.append(m)
            
            if len(good_matches) > 10:
                # Get matched points
                pts_prev = np.float32([self.prev_kp[m.queryIdx].pt for m in good_matches])
                pts_curr = np.float32([kp[m.trainIdx].pt for m in good_matches])
                
                # Compute essential matrix
                K = np.array([[self.focal_length, 0, self.cx],
                              [0, self.focal_length, self.cy],
                              [0, 0, 1]])
                
                E, mask = cv2.findEssentialMat(pts_prev, pts_curr, K, method=cv2.RANSAC, prob=0.999, threshold=1.0)
                
                if E is not None:
                    # Recover pose
                    _, R_delta, t_delta, mask = cv2.recoverPose(E, pts_prev, pts_curr, K)
                    
                    # Update scale with depth if available
                    if depth_map is not None:
                        # Use median depth of matched points for scale estimation
                        depths = []
                        for pt in pts_prev:
                            x, y = int(pt[0]), int(pt[1])
                            if 0 <= y < depth_map.shape[0] and 0 <= x < depth_map.shape[1]:
                                depths.append(depth_map[y, x])
                        if depths:
                            median_depth = np.median(depths)
                            self.scale = median_depth / 10.0  # adaptive scale
                    
                    # Update rotation
                    self.rotation = self.rotation @ R_delta
                    
                    # Update position
                    translation = self.scale * self.rotation @ t_delta.flatten()
                    self.position += translation
        
        # Update previous frame
        self.prev_frame = frame_gray.copy()
        self.prev_kp = kp
        self.prev_desc = desc
        
        return self.position.copy(), self.rotation.copy()
    
    def get_transform_matrix(self):
        """Get 4x4 transformation matrix"""
        T = np.eye(4)
        T[:3, :3] = self.rotation
        T[:3, 3] = self.position
        return T

print("✓ Visual Odometry class defined")

✓ Visual Odometry class defined


## 5. 3D Occupancy Grid Class

Probabilistic voxel grid for obstacle representation

In [4]:
class OccupancyGrid3D:
    """3D voxel-based occupancy grid with probabilistic updates"""
    
    def __init__(self, voxel_size=0.05, grid_size=(200, 200, 100)):
        """
        Args:
            voxel_size: Size of each voxel in meters
            grid_size: Number of voxels in (x, y, z) dimensions
        """
        self.voxel_size = voxel_size
        self.grid_size = grid_size
        
        # Initialize grid with log-odds (0 = unknown, >0 = occupied, <0 = free)
        self.grid = np.zeros(grid_size, dtype=np.float32)
        
        # Grid origin (center of grid)
        self.origin = np.array([grid_size[0] // 2, grid_size[1] // 2, 0])
        
        # Occupancy probability parameters
        self.prob_hit = 0.7  # Probability of hit if occupied
        self.prob_miss = 0.4  # Probability of miss if free
        self.log_odds_hit = np.log(self.prob_hit / (1 - self.prob_hit))
        self.log_odds_miss = np.log(self.prob_miss / (1 - self.prob_miss))
        
        # Thresholds
        self.occupied_thresh = 0.6
        self.free_thresh = 0.4
        
    def world_to_grid(self, points):
        """Convert world coordinates (meters) to grid indices"""
        grid_coords = (points / self.voxel_size + self.origin).astype(np.int32)
        return grid_coords
    
    def grid_to_world(self, indices):
        """Convert grid indices to world coordinates (meters)"""
        world_coords = (indices - self.origin) * self.voxel_size
        return world_coords
    
    def is_valid_index(self, indices):
        """Check if indices are within grid bounds"""
        return np.all((indices >= 0) & (indices < self.grid_size), axis=-1)
    
    def update_from_depth(self, depth_map, camera_position, camera_rotation, 
                          focal_length=500, cx=320, cy=240):
        """
        Update occupancy grid from depth map and camera pose
        
        Args:
            depth_map: HxW depth map (in meters)
            camera_position: 3D position of camera (x, y, z)
            camera_rotation: 3x3 rotation matrix
            focal_length: Camera focal length in pixels
            cx, cy: Camera principal point
        """
        h, w = depth_map.shape
        
        # Downsample for efficiency (every 4th pixel)
        step = 4
        
        for v in range(0, h, step):
            for u in range(0, w, step):
                depth = depth_map[v, u]
                
                if depth <= 0 or depth > 10:  # Ignore invalid or too far
                    continue
                
                # Back-project to 3D camera coordinates
                x_cam = (u - cx) * depth / focal_length
                y_cam = (v - cy) * depth / focal_length
                z_cam = depth
                
                point_cam = np.array([x_cam, y_cam, z_cam])
                
                # Transform to world coordinates
                point_world = camera_rotation @ point_cam + camera_position
                
                # Convert to grid coordinates
                grid_coord = self.world_to_grid(point_world)
                
                if not self.is_valid_index(grid_coord):
                    continue
                
                # Ray tracing from camera to point
                cam_grid = self.world_to_grid(camera_position)
                
                # Bresenham's line algorithm in 3D
                line_points = self.bresenham_3d(cam_grid, grid_coord)
                
                for i, pt in enumerate(line_points):
                    if not self.is_valid_index(pt):
                        continue
                    
                    x, y, z = pt
                    
                    if i < len(line_points) - 1:
                        # Free space along the ray
                        self.grid[x, y, z] += self.log_odds_miss
                        self.grid[x, y, z] = np.clip(self.grid[x, y, z], -10, 10)
                    else:
                        # Occupied at the endpoint
                        self.grid[x, y, z] += self.log_odds_hit
                        self.grid[x, y, z] = np.clip(self.grid[x, y, z], -10, 10)
    
    def bresenham_3d(self, start, end):
        """3D Bresenham's line algorithm"""
        points = []
        
        x0, y0, z0 = start
        x1, y1, z1 = end
        
        dx = abs(x1 - x0)
        dy = abs(y1 - y0)
        dz = abs(z1 - z0)
        
        xs = 1 if x1 > x0 else -1
        ys = 1 if y1 > y0 else -1
        zs = 1 if z1 > z0 else -1
        
        # Driving axis is X
        if dx >= dy and dx >= dz:
            p1 = 2 * dy - dx
            p2 = 2 * dz - dx
            while x0 != x1:
                points.append(np.array([x0, y0, z0]))
                x0 += xs
                if p1 >= 0:
                    y0 += ys
                    p1 -= 2 * dx
                if p2 >= 0:
                    z0 += zs
                    p2 -= 2 * dx
                p1 += 2 * dy
                p2 += 2 * dz
        # Driving axis is Y
        elif dy >= dx and dy >= dz:
            p1 = 2 * dx - dy
            p2 = 2 * dz - dy
            while y0 != y1:
                points.append(np.array([x0, y0, z0]))
                y0 += ys
                if p1 >= 0:
                    x0 += xs
                    p1 -= 2 * dy
                if p2 >= 0:
                    z0 += zs
                    p2 -= 2 * dy
                p1 += 2 * dx
                p2 += 2 * dz
        # Driving axis is Z
        else:
            p1 = 2 * dy - dz
            p2 = 2 * dx - dz
            while z0 != z1:
                points.append(np.array([x0, y0, z0]))
                z0 += zs
                if p1 >= 0:
                    y0 += ys
                    p1 -= 2 * dz
                if p2 >= 0:
                    x0 += xs
                    p2 -= 2 * dz
                p1 += 2 * dy
                p2 += 2 * dx
        
        points.append(np.array([x0, y0, z0]))
        return points
    
    def get_occupied_voxels(self):
        """Get list of occupied voxel coordinates"""
        # Convert log-odds to probability
        prob = 1 / (1 + np.exp(-self.grid))
        occupied_mask = prob > self.occupied_thresh
        occupied_indices = np.argwhere(occupied_mask)
        return occupied_indices
    
    def get_free_voxels(self):
        """Get list of free voxel coordinates"""
        prob = 1 / (1 + np.exp(-self.grid))
        free_mask = prob < self.free_thresh
        free_indices = np.argwhere(free_mask)
        return free_indices
    
    def export_point_cloud(self):
        """Export occupied voxels as point cloud for visualization"""
        occupied = self.get_occupied_voxels()
        world_points = self.grid_to_world(occupied)
        return world_points

print("✓ OccupancyGrid3D class defined")

✓ OccupancyGrid3D class defined


## 6. Depth Estimation Function

In [5]:
def estimate_depth(frame_rgb, model, processor, device, max_depth=10.0):
    """
    Estimate depth from RGB image (works with grayscale converted to RGB)
    
    Args:
        frame_rgb: Input frame (H, W, 3)
        model: Depth model
        processor: Image processor
        device: torch device
        max_depth: Maximum depth in meters
    
    Returns:
        depth_map: Depth map in meters (H, W)
    """
    # Prepare image
    inputs = processor(images=frame_rgb, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth
    
    # Interpolate to original size
    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=frame_rgb.shape[:2],
        mode="bicubic",
        align_corners=False,
    ).squeeze()
    
    # Convert to numpy and normalize to meters
    depth = prediction.cpu().numpy()
    
    # Normalize depth to 0-max_depth meters
    depth_min = depth.min()
    depth_max = depth.max()
    depth_normalized = (depth - depth_min) / (depth_max - depth_min) * max_depth
    
    return depth_normalized

print("✓ Depth estimation function defined")

✓ Depth estimation function defined


## 7. Visualization Functions

In [6]:
def visualize_depth(depth_map):
    """Colorize depth map for visualization"""
    depth_colored = cv2.applyColorMap(
        (depth_map / depth_map.max() * 255).astype(np.uint8),
        cv2.COLORMAP_TURBO
    )
    return depth_colored

def draw_trajectory(frame, vo, scale=30):
    """Draw trajectory on frame"""
    # Draw current position
    pos_2d = (int(frame.shape[1] // 2 + vo.position[0] * scale),
              int(frame.shape[0] // 2 + vo.position[2] * scale))
    cv2.circle(frame, pos_2d, 5, (0, 255, 0), -1)
    
    # Draw axes showing orientation
    axis_length = 30
    origin = (frame.shape[1] // 2, frame.shape[0] // 2)
    
    # X axis (red)
    x_end = (int(origin[0] + vo.rotation[0, 0] * axis_length),
             int(origin[1] + vo.rotation[2, 0] * axis_length))
    cv2.arrowedLine(frame, origin, x_end, (0, 0, 255), 2)
    
    # Z axis (blue) - forward
    z_end = (int(origin[0] + vo.rotation[0, 2] * axis_length),
             int(origin[1] + vo.rotation[2, 2] * axis_length))
    cv2.arrowedLine(frame, origin, z_end, (255, 0, 0), 2)
    
    return frame

print("✓ Visualization functions defined")

✓ Visualization functions defined


## 8. Main Processing Loop

Real-time spatial mapping with webcam (simulating B&W drone camera)

In [11]:
# Initialize components
vo = VisualOdometry()
occupancy_grid = OccupancyGrid3D(voxel_size=0.05, grid_size=(200, 200, 100))

# Open webcam
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

# FPS tracking
fps_history = deque(maxlen=30)
frame_count = 0
update_grid_every = 3  # Update occupancy grid every N frames for performance

print("Starting spatial mapping...")
print("Controls:")
print("  'q' - Quit")
print("  's' - Save occupancy grid")
print("  'r' - Reset mapping")
print("\\nMove the camera slowly around the room...")

try:
    while True:
        start_time = time.time()
        
        ret, frame = cap.read()
        if not ret:
            print("Failed to read from camera")
            break
        
        # Convert to grayscale (simulating B&W camera)
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Convert grayscale to RGB for depth model
        frame_rgb = cv2.cvtColor(frame_gray, cv2.COLOR_GRAY2RGB)
        
        # Estimate depth
        depth_map = estimate_depth(frame_rgb, depth_model, image_processor, device)
        
        # Update visual odometry
        camera_pos, camera_rot = vo.update(frame_gray, depth_map)
        
        # Update occupancy grid (every N frames)
        if frame_count % update_grid_every == 0:
            occupancy_grid.update_from_depth(
                depth_map, 
                camera_pos, 
                camera_rot,
                focal_length=vo.focal_length,
                cx=vo.cx,
                cy=vo.cy
            )
        
        # Visualization
        depth_colored = visualize_depth(depth_map)
        
        # Create display frame
        display_frame = frame.copy()
        display_frame = draw_trajectory(display_frame, vo)
        
        # Add info text
        end_time = time.time()
        fps = 1.0 / (end_time - start_time)
        fps_history.append(fps)
        avg_fps = np.mean(fps_history)
        
        occupied_voxels = len(occupancy_grid.get_occupied_voxels())
        
        cv2.putText(display_frame, f"FPS: {avg_fps:.1f}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display_frame, f"Position: ({camera_pos[0]:.2f}, {camera_pos[1]:.2f}, {camera_pos[2]:.2f})", 
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(display_frame, f"Occupied Voxels: {occupied_voxels}", 
                    (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Stack displays
        top_row = np.hstack([display_frame, cv2.cvtColor(depth_colored, cv2.COLOR_BGR2RGB)])
        
        # Show
        cv2.imshow('Spatial Mapping - Camera + Depth', top_row)
        
        # Handle keys
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            print("\\nSaving occupancy grid...")
            point_cloud = occupancy_grid.export_point_cloud()
            np.save('occupancy_grid.npy', occupancy_grid.grid)
            np.save('occupied_points.npy', point_cloud)
            print(f"✓ Saved {len(point_cloud)} occupied voxels")
        elif key == ord('r'):
            print("\\nResetting mapping...")
            vo = VisualOdometry()
            occupancy_grid = OccupancyGrid3D(voxel_size=0.05, grid_size=(200, 200, 100))
        
        frame_count += 1
        
except KeyboardInterrupt:
    print("\\nStopped by user")
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("\\nCleaned up resources")

Starting spatial mapping...
Controls:
  'q' - Quit
  's' - Save occupancy grid
  'r' - Reset mapping
\nMove the camera slowly around the room...
\nCleaned up resources


## 9. 3D Visualization of Occupancy Grid (Last Stopped Here)

Visualize the mapped environment in 3D

In [13]:
def visualize_occupancy_grid_3d(occupancy_grid, camera_trajectory=None):
    """
    Visualize 3D occupancy grid using Open3D
    
    Args:
        occupancy_grid: OccupancyGrid3D instance
        camera_trajectory: List of camera positions (optional)
    """
    # Get occupied voxels
    occupied_points = occupancy_grid.export_point_cloud()
    
    if len(occupied_points) == 0:
        print("No occupied voxels to visualize")
        return
    
    print(f"Visualizing {len(occupied_points)} occupied voxels...")
    
    # Create point cloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(occupied_points)
    
    # Color by height (Z coordinate)
    colors = np.zeros_like(occupied_points)
    z_vals = occupied_points[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    if z_max > z_min:
        z_normalized = (z_vals - z_min) / (z_max - z_min)
    else:
        z_normalized = np.zeros_like(z_vals)
    
    # Turbo colormap
    colors[:, 0] = z_normalized  # R
    colors[:, 1] = 1 - z_normalized  # G
    colors[:, 2] = 0.5  # B
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # Create coordinate frame at origin
    coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
        size=0.5, origin=[0, 0, 0]
    )
    
    geometries = [pcd, coordinate_frame]
    
    # Add camera trajectory if provided
    if camera_trajectory is not None and len(camera_trajectory) > 1:
        points = np.array(camera_trajectory)
        lines = [[i, i+1] for i in range(len(points)-1)]
        
        line_set = o3d.geometry.LineSet()
        line_set.points = o3d.utility.Vector3dVector(points)
        line_set.lines = o3d.utility.Vector2iVector(lines)
        line_set.colors = o3d.utility.Vector3dVector([[1, 0, 0] for _ in lines])  # Red trajectory
        
        geometries.append(line_set)
    
    # Visualize
    o3d.visualization.draw_geometries(
        geometries,
        window_name="3D Occupancy Grid",
        width=1280,
        height=720,
        left=50,
        top=50,
        point_show_normal=False
    )

# Run visualization (if you've run the mapping above)
if 'occupancy_grid' in locals():
    visualize_occupancy_grid_3d(occupancy_grid)
else:
    print("Run the mapping loop first to generate occupancy grid")

Visualizing 12300 occupied voxels...


## 10. Flight Control Utilities

Functions to query occupancy grid for safe flight paths

In [ ]:
class FlightController:
    """Simple flight controller using occupancy grid"""
    
    def __init__(self, occupancy_grid, safety_margin=0.3):
        """
        Args:
            occupancy_grid: OccupancyGrid3D instance
            safety_margin: Safety margin around obstacles in meters
        """
        self.grid = occupancy_grid
        self.safety_margin = safety_margin
        self.safety_voxels = int(safety_margin / occupancy_grid.voxel_size)
    
    def is_position_safe(self, position):
        """
        Check if a 3D position is safe (not occupied or too close to obstacles)
        
        Args:
            position: (x, y, z) in meters
        
        Returns:
            bool: True if safe, False if occupied or too close to obstacle
        """
        grid_pos = self.grid.world_to_grid(position)
        
        if not self.grid.is_valid_index(grid_pos):
            return False  # Outside grid bounds
        
        # Check if position itself is occupied
        x, y, z = grid_pos
        prob = 1 / (1 + np.exp(-self.grid.grid[x, y, z]))
        
        if prob > self.grid.occupied_thresh:
            return False
        
        # Check safety margin (simplified - check cube around position)
        for dx in range(-self.safety_voxels, self.safety_voxels + 1):
            for dy in range(-self.safety_voxels, self.safety_voxels + 1):
                for dz in range(-self.safety_voxels, self.safety_voxels + 1):
                    check_pos = grid_pos + np.array([dx, dy, dz])
                    
                    if not self.grid.is_valid_index(check_pos):
                        continue
                    
                    cx, cy, cz = check_pos
                    prob = 1 / (1 + np.exp(-self.grid.grid[cx, cy, cz]))
                    
                    if prob > self.grid.occupied_thresh:
                        return False  # Too close to obstacle
        
        return True
    
    def get_safe_directions(self, current_position, directions):
        """
        Check which directions are safe to move
        
        Args:
            current_position: Current (x, y, z) in meters
            directions: List of direction vectors to check (e.g., [[1,0,0], [0,1,0], ...])
        
        Returns:
            List of safe directions
        """
        safe_dirs = []
        
        for direction in directions:
            # Check position 0.5m ahead in this direction
            test_position = current_position + 0.5 * np.array(direction)
            
            if self.is_position_safe(test_position):
                safe_dirs.append(direction)
        
        return safe_dirs
    
    def find_nearest_obstacle(self, position, max_distance=5.0):
        """
        Find nearest obstacle to given position
        
        Args:
            position: (x, y, z) in meters
            max_distance: Maximum search distance in meters
        
        Returns:
            (distance, obstacle_position) or (None, None) if no obstacle found
        """
        grid_pos = self.grid.world_to_grid(position)
        max_voxels = int(max_distance / self.grid.voxel_size)
        
        occupied_voxels = self.grid.get_occupied_voxels()
        
        if len(occupied_voxels) == 0:
            return None, None
        
        # Calculate distances to all occupied voxels
        distances = np.linalg.norm(occupied_voxels - grid_pos, axis=1)
        
        # Find nearest
        min_idx = np.argmin(distances)
        min_distance_voxels = distances[min_idx]
        
        if min_distance_voxels > max_voxels:
            return None, None
        
        min_distance_meters = min_distance_voxels * self.grid.voxel_size
        nearest_obstacle = self.grid.grid_to_world(occupied_voxels[min_idx])
        
        return min_distance_meters, nearest_obstacle
    
    def get_flight_clearance_map(self, height, grid_resolution=0.1):
        """
        Generate 2D clearance map at a specific height
        Useful for path planning at a fixed altitude
        
        Args:
            height: Z height in meters
            grid_resolution: Resolution of output map in meters
        
        Returns:
            clearance_map: 2D array where values represent distance to nearest obstacle
        """
        # Create 2D grid at specified height
        x_range = np.arange(-5, 5, grid_resolution)
        y_range = np.arange(-5, 5, grid_resolution)
        
        clearance_map = np.zeros((len(y_range), len(x_range)))
        
        for i, y in enumerate(y_range):
            for j, x in enumerate(x_range):
                position = np.array([x, y, height])
                distance, _ = self.find_nearest_obstacle(position)
                
                if distance is None:
                    clearance_map[i, j] = 10.0  # No obstacle nearby
                else:
                    clearance_map[i, j] = distance
        
        return clearance_map, x_range, y_range

# Example usage
if 'occupancy_grid' in locals():
    flight_controller = FlightController(occupancy_grid, safety_margin=0.3)
    
    # Test current position safety
    test_pos = np.array([1.0, 0.5, 1.5])
    is_safe = flight_controller.is_position_safe(test_pos)
    print(f"\\nPosition {test_pos} is {'SAFE' if is_safe else 'UNSAFE'}")
    
    # Test directions
    directions = [
        [1, 0, 0],   # Forward
        [-1, 0, 0],  # Backward
        [0, 1, 0],   # Right
        [0, -1, 0],  # Left
        [0, 0, 1],   # Up
        [0, 0, -1]   # Down
    ]
    
    safe_dirs = flight_controller.get_safe_directions(test_pos, directions)
    print(f"Safe directions from {test_pos}: {len(safe_dirs)}/{len(directions)}")
    
    # Find nearest obstacle
    distance, obs_pos = flight_controller.find_nearest_obstacle(test_pos)
    if distance:
        print(f"Nearest obstacle: {distance:.2f}m away at {obs_pos}")
    else:
        print("No obstacles detected nearby")
else:
    print("Run the mapping loop first to generate occupancy grid")

## 11. Visualize Flight Clearance Map

2D clearance map showing safe flying zones at a specific altitude

In [ ]:
if 'flight_controller' in locals():
    # Generate clearance map at 1.5m height (typical drone flying height)
    print("Generating flight clearance map at 1.5m altitude...")
    clearance_map, x_range, y_range = flight_controller.get_flight_clearance_map(
        height=1.5, 
        grid_resolution=0.1
    )
    
    # Plot clearance map
    plt.figure(figsize=(12, 10))
    
    im = plt.imshow(
        clearance_map, 
        extent=[x_range[0], x_range[-1], y_range[0], y_range[-1]],
        origin='lower',
        cmap='RdYlGn',
        vmin=0,
        vmax=3
    )
    
    plt.colorbar(im, label='Distance to Obstacle (m)')
    plt.xlabel('X (meters)')
    plt.ylabel('Y (meters)')
    plt.title('Flight Clearance Map at 1.5m Altitude\\n(Green=Safe, Yellow=Caution, Red=Danger)')
    plt.grid(True, alpha=0.3)
    
    # Mark safe zones
    safe_threshold = 0.5  # meters
    safe_mask = clearance_map > safe_threshold
    safe_percentage = (np.sum(safe_mask) / safe_mask.size) * 100
    
    plt.text(0.02, 0.98, f'Safe Area: {safe_percentage:.1f}%', 
             transform=plt.gca().transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
             verticalalignment='top')
    
    plt.tight_layout()
    plt.savefig('flight_clearance_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ {safe_percentage:.1f}% of mapped area is safe for flight")
else:
    print("Run the mapping and flight controller initialization first")

## 12. Crazyflie 2.0 Integration Guide

### Hardware Setup
1. **Crazyflie 2.0** with flow deck (for altitude/position hold)
2. **AI Deck** (GAP8 + ESP32) for camera streaming
3. **B&W Camera** connected to AI deck

### Software Architecture
```
┌─────────────────┐
│  Crazyflie 2.0  │
│   + AI Deck     │
│   + B&W Camera  │
└────────┬────────┘
         │ WiFi/Radio
         │ Video Stream (30 FPS)
         ▼
┌─────────────────┐
│  Cloud Server   │
│   (This Code)   │
│  - Depth Est.   │
│  - Mapping      │
│  - Path Plan    │
└────────┬────────┘
         │ Control Commands
         ▼
┌─────────────────┐
│  Flight Ctrl    │
│  (Crazyflie)    │
└─────────────────┘
```

### Camera Stream Setup (AI Deck)

The AI Deck streams JPEG frames over WiFi. Below is example code to capture the stream:

In [ ]:
# Crazyflie AI Deck Camera Stream Receiver
# This replaces cv2.VideoCapture(0) in the main loop

import socket
import struct

class CrazyflieCamera:
    """Receive camera stream from Crazyflie AI deck"""
    
    def __init__(self, ip='192.168.4.1', port=5000):
        """
        Args:
            ip: AI deck IP address (default: 192.168.4.1)
            port: Streaming port (default: 5000)
        """
        self.ip = ip
        self.port = port
        self.sock = None
        self.connected = False
    
    def connect(self):
        """Connect to AI deck"""
        try:
            self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.sock.connect((self.ip, self.port))
            self.connected = True
            print(f"✓ Connected to Crazyflie AI deck at {self.ip}:{self.port}")
        except Exception as e:
            print(f"✗ Failed to connect: {e}")
            self.connected = False
    
    def read(self):
        """Read frame from stream"""
        if not self.connected:
            return False, None
        
        try:
            # Receive frame size (4 bytes)
            size_data = self.sock.recv(4)
            if len(size_data) < 4:
                return False, None
            
            frame_size = struct.unpack('<I', size_data)[0]
            
            # Receive frame data
            frame_data = b''
            while len(frame_data) < frame_size:
                chunk = self.sock.recv(frame_size - len(frame_data))
                if not chunk:
                    return False, None
                frame_data += chunk
            
            # Decode JPEG
            nparr = np.frombuffer(frame_data, np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)
            
            return True, frame
            
        except Exception as e:
            print(f"Error reading frame: {e}")
            return False, None
    
    def release(self):
        """Close connection"""
        if self.sock:
            self.sock.close()
            self.connected = False

# Example usage (when connected to actual Crazyflie)
# Replace the webcam initialization in section 8 with:
#
# cf_camera = CrazyflieCamera(ip='192.168.4.1')
# cf_camera.connect()
#
# And replace cap.read() with:
# ret, frame_gray = cf_camera.read()
# frame = cv2.cvtColor(frame_gray, cv2.COLOR_GRAY2BGR)  # For visualization

print("✓ CrazyflieCamera class defined")

### Flight Control Integration

Send navigation commands back to Crazyflie based on occupancy grid:

In [ ]:
# Crazyflie Flight Control Integration
# Install: pip install cflib

"""
import cflib.crtp
from cflib.crazyflie import Crazyflie
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie

class CrazyflieNavigator:
    def __init__(self, uri='radio://0/80/2M/E7E7E7E7E7'):
        self.uri = uri
        self.scf = None
        
    def connect(self):
        cflib.crtp.init_drivers()
        self.scf = SyncCrazyflie(self.uri, cf=Crazyflie(rw_cache='./cache'))
        self.scf.open_link()
        print(f"✓ Connected to Crazyflie at {self.uri}")
    
    def send_velocity_command(self, vx, vy, vz, yaw_rate):
        '''
        Send velocity command to Crazyflie
        
        Args:
            vx, vy, vz: Velocity in m/s (body frame)
            yaw_rate: Yaw rate in deg/s
        '''
        if self.scf:
            self.scf.cf.commander.send_velocity_world_setpoint(vx, vy, vz, yaw_rate)
    
    def navigate_safe(self, flight_controller, target_direction):
        '''
        Navigate in target direction while avoiding obstacles
        
        Args:
            flight_controller: FlightController instance
            target_direction: Desired direction [x, y, z]
        '''
        # Get current position (would come from external positioning system)
        current_pos = np.array([0, 0, 1.5])  # Example
        
        # Check if target direction is safe
        is_safe = flight_controller.is_position_safe(
            current_pos + 0.5 * np.array(target_direction)
        )
        
        if is_safe:
            # Move in target direction
            velocity = 0.3  # m/s
            vx = target_direction[0] * velocity
            vy = target_direction[1] * velocity
            vz = target_direction[2] * velocity
            self.send_velocity_command(vx, vy, vz, 0)
        else:
            # Find alternative safe direction
            directions = [
                [1, 0, 0], [-1, 0, 0],
                [0, 1, 0], [0, -1, 0],
                [0, 0, 1], [0, 0, -1]
            ]
            safe_dirs = flight_controller.get_safe_directions(current_pos, directions)
            
            if safe_dirs:
                # Move in first safe direction
                direction = safe_dirs[0]
                velocity = 0.3
                vx = direction[0] * velocity
                vy = direction[1] * velocity
                vz = direction[2] * velocity
                self.send_velocity_command(vx, vy, vz, 0)
            else:
                # No safe direction - hover
                self.send_velocity_command(0, 0, 0, 0)
                print("⚠ No safe direction - hovering")
    
    def disconnect(self):
        if self.scf:
            self.scf.close_link()

# Example usage:
# navigator = CrazyflieNavigator()
# navigator.connect()
# navigator.navigate_safe(flight_controller, target_direction=[1, 0, 0])
# navigator.disconnect()
"""

print("✓ Crazyflie integration code ready")